In [1]:
import numpy as np
import torch

SEED = 1234
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# 0. Choosing "teacher" model

In [2]:
checkpoint = "bert-base-uncased"
task_name  = "mrpc"

# 1. Loading our mrpc part of the GLUE dataset

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue", task_name)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating test split: 100%|██████████| 1725/1725 [00:00<00:00, 735506.19 examples/s]


In [4]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [5]:
raw_datasets['train']

Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 3668
})

In [6]:
from datasets import DatasetDict

In [7]:
raw_datasets = DatasetDict({
    "train": raw_datasets['train'],
    "validation": raw_datasets['validation'],
    "test": raw_datasets['test']
})

In [8]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [9]:
raw_train_dataset = raw_datasets["train"]
raw_train_dataset[0]

{'sentence1': 'Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .',
 'sentence2': 'Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .',
 'label': 1,
 'idx': 0}

In [10]:
raw_train_dataset[5]['sentence1'], raw_train_dataset[5]['sentence2']

('Revenue in the first quarter of the year dropped 15 percent from the same period a year earlier .',
 "With the scandal hanging over Stewart 's company , revenue the first quarter of the year dropped 15 percent from the same period a year earlier .")

In [11]:
raw_train_dataset[5]['label']

1

In [12]:
raw_train_dataset[5]['idx']

5

In [13]:
raw_train_dataset.features

{'sentence1': Value('string'),
 'sentence2': Value('string'),
 'label': ClassLabel(names=['not_equivalent', 'equivalent']),
 'idx': Value('int32')}

# 2. Preprocess

In [14]:
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets

Map: 100%|██████████| 1725/1725 [00:00<00:00, 38970.02 examples/s]


DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

# 3. Preparing for Training

In [15]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [16]:
tokenized_datasets = tokenized_datasets.remove_columns(["sentence1", "sentence2", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")
tokenized_datasets["train"].column_names

['labels', 'input_ids', 'token_type_ids', 'attention_mask']

In [17]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=32, collate_fn=data_collator
)

val_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=32, collate_fn=data_collator
)

eval_dataloader = DataLoader(
    tokenized_datasets["test"], batch_size=32, collate_fn=data_collator
)

In [18]:
for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}

{'labels': torch.Size([32]),
 'input_ids': torch.Size([32, 100]),
 'token_type_ids': torch.Size([32, 100]),
 'attention_mask': torch.Size([32, 100])}

# 4. Load Model

In [19]:
import sys
sys.path.insert(0, '../../')

In [20]:
from Bert_model.modeling_bert import BertForSequenceClassification

In [21]:
# id2label, label2id dicts for the outputs for the model
labels = tokenized_datasets["train"].features["labels"].names
num_labels = len(labels)
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = str(i)
    id2label[str(i)] = label

In [22]:
teacher_model = BertForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    output_hidden_states=True,
    output_attentions=True
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [23]:
teacher_model.set_use_module_grafting(False)
teacher_model.set_use_scc_status(False)

In [24]:
outputs = teacher_model(**batch)
print(outputs.loss, outputs.logits.shape)

tensor(1.2996, grad_fn=<NllLossBackward0>) torch.Size([32, 2])


In [25]:
device = torch.device("cuda:1") if torch.cuda.is_available() else torch.device("cpu")
teacher_model.to(device)

device

device(type='cuda', index=1)

### Load Trained Weights

In [26]:
load_path = '../../glue_fine_tune/weights/'
best_weight = torch.load(load_path + f'bert-{task_name}.pt', map_location=device)

In [27]:
teacher_model.load_state_dict(best_weight['model_state_dict'])

<All keys matched successfully>

In [28]:
from train_eval_func import eval_loop

In [29]:
eval_loop(teacher_model, val_dataloader, task_name, device)[0]

{'accuracy': 0.8676470588235294, 'f1': 0.9093959731543624}

In [43]:
from train_eval_func import train
import os
from torch.optim import AdamW

In [45]:
def follow_up_train(
    model, load_obj, train_dataloader, val_dataloader, device, recovery_epochs, task_name, lr_rate, 
    patience, alpha, temperature, save_dir='./weights/', teacher_model=False
):
    if 'post_train' in load_obj:
        if load_obj['post_train']:
            print("✓ Model already has post-training - skipping")
            return False
    if load_obj['is_retrained']:
        pprint("✓ Model was retrained during merge - skipping")
        return False

    print("→ Model needs follow-up training")
    print(f"  Task: {task_name}")
    print(f"  Epochs: {recovery_epochs}")
    print(f"  Learning rate: {lr_rate}")

    # Evaluate before training
    print("\n[1/4] Evaluating before training...")
    metrics_before = eval_loop(model, val_dataloader, task_name, device)[0]
    print("  Metrics before training:")
    for metric_name, value in metrics_before.items():
        print(f"    {metric_name}: {value:.4f}")

    # Create checkpoint directory
    recovery_dir = os.path.join(save_dir, 'followup_training')
    os.makedirs(recovery_dir, exist_ok=True)

    # Setup optimizer
    print("\n[2/4] Setting up training...")
    optimizer = AdamW(model.parameters(), lr=lr_rate)
    
    # Temporary checkpoint path
    temp_save_path = os.path.join(
        recovery_dir, 
        f'temp_followup_{task_name}.pt'
    )
    
    # Train with checkpoint saving
    print("\n[3/4] Training...")
    from train_eval_func import train
    
    train_stats = train(
        model,
        train_dataloader,
        val_dataloader,
        task_name,
        device,
        num_epochs=recovery_epochs,
        lr=lr_rate,
        patience=patience,
        use_early_stopping=True,
        optimizer=optimizer,
        lr_scheduler=None,
        save_path=temp_save_path,
        display_epoch_iter=True,
        teacher_model=teacher_model,
        alpha=alpha,
        temperature=temperature
    )
    

# 5. Check for Layer 8 Train

In [32]:
load_path = f'./weights'

In [33]:
load_obj = torch.load(load_path + f'/layer-8-{task_name}.pt', map_location=device)

In [35]:
load_obj['is_retrained']

False

In [36]:
load_obj.keys()

dict_keys(['model_state_dict', 'layer_track', 'train_stats', 'metrics_after_merge', 'metrics_after_recovery', 'merge_iteration', 'num_layers', 'is_retrained'])

In [ ]:
def follow_up_train(
    model, load_obj, train_dataloader, val_dataloader, device, recovery_epochs, task_name, lr_rate, 
    patience, alpha, temperature, save_dir='./weights/', teacher_model=False
):

In [39]:
recovery_epochs=5
lr_rate=1e-5
patience = 3
alpha=0.3
temperature=7

In [46]:
follow_up_train(teacher_model, load_obj, train_dataloader, val_dataloader, device, 
                recovery_epochs, task_name, lr_rate, patience, alpha, temperature,
               teacher_model=teacher_model)

→ Model needs follow-up training
  Task: mrpc
  Epochs: 5
  Learning rate: 1e-05

[1/4] Evaluating before training...
  Metrics before training:
    accuracy: 0.8676
    f1: 0.9094

[2/4] Setting up training...

[3/4] Training...


Training:   0%|          | 0/575 [00:00<?, ?it/s]

Knowledge distillation used!


/home/abhinavl/work/Layer_Graft/layer-grafting-publication/similar_layer_merge/with_kd/../../train_eval_func.py:223: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:835.)
  total_loss += float(loss)
Training:  20%|██        | 115/575 [00:19<00:59,  7.70it/s]

✓ Saved checkpoint (best loss: 0.4414)

<----------------- Epoch 1 ----------------->
Loss: 0.12, Training Metrics:
  accuracy: 0.9695
  f1: 0.9771
Validation Loss: 0.44, Validation Metrics:
  accuracy: 0.8529
  f1: 0.9010
Elapsed Time: 20.0688 sec


...


<----------------- Epoch 2 ----------------->
Loss: 0.06, Training Metrics:
  accuracy: 0.9741
  f1: 0.9807
Validation Loss: 0.47, Validation Metrics:
  accuracy: 0.8529
  f1: 0.9003
Elapsed Time: 18.1628 sec


Training:  60%|██████    | 345/575 [01:07<00:31,  7.28it/s]


<----------------- Epoch 3 ----------------->
Loss: 0.05, Training Metrics:
  accuracy: 0.9815
  f1: 0.9862
Validation Loss: 0.5, Validation Metrics:
  accuracy: 0.8554
  f1: 0.8985
Elapsed Time: 18.4276 sec


Training:  80%|████████  | 460/575 [01:27<00:21,  5.27it/s]


<----------------- Epoch 4 ----------------->
Loss: 0.05, Training Metrics:
  accuracy: 0.9774
  f1: 0.9832
Validation Loss: 0.42, Validation Metrics:
  accuracy: 0.8627
  f1: 0.9007
Elapsed Time: 18.4543 sec
>>>>> Early stopping callback <<<<<

Total Training Time: 87.23 sec
Chosen Primary Metric: f1
Best Metric Run: 0.9010


In [46]:
load_obj = torch.load(load_path + f'/layer-7-{task_name}.pt', map_location=device)

In [47]:
follow_up_train(teacher_model, device, load_obj)

No need to train further!
